# AGC_Trollheim: reduced-model forensic check vs nonlinear OpenHPL
This notebook accompanies `OpenHPL.Examples.AGC_Trollheim`. It computes the transferred inertia constant and an approximate water starting time, simulates a classical reduced hydro-governor model, then compares it with the Railway/OpenModelica CSV result.


In [ ]:
import pathlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
PBASE=150e6; F0=50.; poles=12; JTOTAL=2e5
omega_m=2*np.pi*F0/(poles/2)
H=0.5*JTOTAL*omega_m**2/PBASE
Q0=23.178260892; g=9.81; Hhyd=340.
A6=np.pi*6**2/4; A4=np.pi*4**2/4
Tw=Q0/(g*Hhyd)*(500/A6+500/A4+600/A6)
print(f'H = {H:.4f} s')
print(f'Approximate Tw = {Tw:.4f} s')


## Reduced hydro model
The classical turbine approximation is
$$G_h(s)=\frac{1-T_ws}{1+0.5T_ws}.$$
With a first-order servo and droop-plus-integral controller, the reduced model is used only as a pre-check; the OpenHPL model remains the nonlinear reference.


In [ ]:
R=.50; Ki=.10; Tg=.30; STEP=5.; DPL=.10; g0=.446339858
def rhs(t,x):
    df,dgate,xh,xi=x
    dload=0. if t<STEP else DPL
    gate_cmd=np.clip(-df/R+Ki*xi,0.05-g0,1.0-g0)
    dPm=3*xh-2*dgate
    return [(dPm-dload)/(2*H),(gate_cmd-dgate)/Tg,(dgate-xh)/(0.5*Tw),-df]
sol=solve_ivp(rhs,(0,65),[0,0,0,0],max_step=.01,rtol=1e-9,atol=1e-11,dense_output=True)
ta=np.linspace(0,65,6501); xa=sol.sol(ta)
fa=F0*(1+xa[0]); pma=(0.5+3*xa[2]-2*xa[1])*PBASE/1e6
post=ta>=5; ia=np.where(post)[0][np.argmin(fa[post])]
print(f'Reduced nadir = {fa[ia]:.4f} Hz at {ta[ia]:.2f} s')
print(f'Reduced f(65) = {fa[-1]:.4f} Hz')


In [ ]:
RES=pathlib.Path('/workspace/results/AGC_Trollheim_res.csv')
if not RES.exists():
    raise FileNotFoundError('Run railway/run_trollheim.sh first')
df=pd.read_csv(RES)
df['Pm_MW']=df['mechanicalPower']/1e6; df['PL_MW']=df['electricalLoad']/1e6
pre=df[df.time<5]; postdf=df[df.time>=5]
i=postdf['frequency_Hz'].idxmin()
summary=pd.DataFrame({
 'metric':['pre f range [Hz]','pre Pm range [MW]','nadir [Hz]','nadir time [s]','f(65) [Hz]','Pm(65) [MW]'],
 'value':[pre.frequency_Hz.max()-pre.frequency_Hz.min(),(pre.Pm_MW.max()-pre.Pm_MW.min()),df.loc[i,'frequency_Hz'],df.loc[i,'time'],df.iloc[-1].frequency_Hz,df.iloc[-1].Pm_MW]})
display(summary)


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df.time,df.frequency_Hz,label='OpenHPL nonlinear')
plt.plot(ta,fa,'--',label='Reduced hydro model')
plt.axvline(5,ls=':'); plt.axhline(50,ls=':')
plt.xlabel('Time [s]'); plt.ylabel('Frequency [Hz]'); plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig('/workspace/results/agc_trollheim_frequency_compare.png',dpi=180)
plt.show()
plt.figure(figsize=(9,5))
plt.plot(df.time,df.Pm_MW,label='OpenHPL turbine power')
plt.plot(df.time,df.PL_MW,label='Electrical load')
plt.plot(ta,pma,'--',label='Reduced mechanical power')
plt.axvline(5,ls=':'); plt.xlabel('Time [s]'); plt.ylabel('Power [MW]'); plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig('/workspace/results/agc_trollheim_power_compare.png',dpi=180)
plt.show()
plt.figure(figsize=(9,5))
plt.plot(df.time,df.guideVane,label='Guide vane [pu]')
plt.plot(df.time,df.turbineFlow,label='Turbine flow [m3/s]')
plt.axvline(5,ls=':'); plt.xlabel('Time [s]'); plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig('/workspace/results/agc_trollheim_gate_flow.png',dpi=180)
plt.show()
